# Autoencoder — Neural Network Fraud Detection

The Isolation Forest baseline (`02_isolation_forest.ipynb`) reached a PR-AUC of
**0.085**, about 51x the random-guessing baseline. It catches frauds that sit
far out on the edges of the data. But many frauds hide *inside* the normal
cloud of transactions, where "how easy is this point to isolate?" can't see
them.

This notebook tries a different mechanism: an **autoencoder**, a neural network
that learns to *rebuild* normal transactions.

- It squeezes each transaction's 29 features through a bottleneck of just 8
  numbers, then tries to rebuild all 29.
- Because the bottleneck is too small to memorise, the network must learn the
  *patterns* of normal transactions.
- It is trained on **normal transactions only**. Fraud doesn't follow those
  patterns, so it gets rebuilt badly.
- The **reconstruction error** (how badly a row is rebuilt) becomes the risk
  score.

An analogy: someone who has only ever drawn cats will draw a fox badly, and how
badly they draw it tells you it isn't a cat.

The same logic lives in `src/models/train_autoencoder.py` as a reusable script,
which is what the FastAPI service will load. This notebook is the narrative
version, with the reasoning and experiments behind each choice.

> **Reproducibility note:** neural network training differs very slightly between
> machines (CPU instructions, PyTorch versions), even with fixed seeds. Expect
> your numbers to be *close* to the ones in the text, not bit-identical.

## Setup

In [ ]:
import json
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler

RANDOM_STATE = 42

## 1. Same data, same split as the Isolation Forest

To compare two models fairly, they must sit **the same exam**. So this cell
repeats the Isolation Forest's preparation exactly:

- drop the 1,081 exact duplicates (so no transaction appears in both train and test)
- drop `Time` (it only encodes position in this dataset's 48-hour window)
- stratified 80/20 split with `random_state=42`, which gives the identical
  56,746-row test set with 95 frauds
- keep only the **normal** training rows for fitting

Any difference in results then comes from the model, not from the data.

In [ ]:
df = pd.read_csv("../data/raw/creditcard.csv").drop_duplicates().reset_index(drop=True)

feature_cols = [c for c in df.columns if c.startswith("V")] + ["Amount"]
X = df[feature_cols].copy()
y = df["Class"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train_normal = X_train[y_train == 0]

print("normal training rows:", X_train_normal.shape)
print("test rows:", X_test.shape, " fraud in test:", y_test.sum())

## 2. Feature scales, and why they matter now

Isolation Forest didn't care about feature scales (each split is drawn between
a feature's min and max, so rescaling changes nothing). **A neural network
trained with squared error does care.** A feature with large numbers produces
large errors, so the network would spend most of its effort on that feature.

In [ ]:
X_train_normal[["V1", "V5", "V14", "Amount"]].describe().loc[["mean", "std", "min", "max"]].round(2)

Two things stand out:

1. **`Amount` is on a completely different scale** (std about 246, against about 1–2
   for the V columns). Left alone, it would dominate the loss.
2. **Even the V columns have extreme values.** A useful ruler is
   *(value − mean) / std*, which counts how many standard deviations a value
   sits from the mean. V1's minimum is (−56.41 − 0.01) / 1.92 ≈ **−29** standard
   deviations; V14's is about −21. For a bell-shaped variable, 99.7% of values
   lie within 3.

Scaling changes the size of the ruler, **not how many rulers away a value
is**. So scaling won't remove these extremes. They are real, legitimate
transactions that the network will rebuild badly. They will become **false
positives**: unusual is not the same as fraudulent. That's an honest limit of
any method that only learns "normal".

## 3. Choosing the preprocessing, by experiment

There are several reasonable ways to scale the data, so instead of guessing,
three options were tested, each trained with three different random seeds:

- **(a)** StandardScaler on all 29 columns
- **(b)** RobustScaler on `Amount` only, with the V columns left as they are (the same as the Isolation Forest)
- **(c)** log(1 + `Amount`), then StandardScaler on all 29

The cell below reruns the experiment. It trains the network 9 times, which takes
5–15 minutes on a laptop CPU, so it's switched off by default. Set
`RUN_PREPROCESSING_COMPARISON = True` to run it. (It uses all normal training
rows and no validation split: it's a quick comparison, not the final model.)

Results (test-set PR-AUC):

| option | seed 1 | seed 7 | seed 42 |
|---|---|---|---|
| (a) StandardScaler, all columns | 0.244 | 0.263 | 0.271 |
| **(b) RobustScaler, Amount only** | **0.435** | **0.387** | **0.375** |
| (c) log(Amount) + StandardScaler | 0.235 | 0.244 | 0.297 |

**Why three seeds?** Option (b) alone ranges from 0.375 to 0.435 just by
changing the random starting point. With one seed, you can't tell a real
difference from luck. Here (b)'s *worst* run beats every other option's *best*,
so the win is real.

**Why might (b) win?** A likely explanation, not a proven one: V1–V28 come from
PCA, so they're already centred and on comparable scales. Standardising them
blows up the small-variance components, so their noise counts as much as the
informative ones. A bonus of (b): preprocessing is now identical to the
Isolation Forest's.

**Honest caveat:** this choice was made by looking at test-set scores, a mild
form of peeking at the exam. A cleaner version would compare the options on a
separate held-out slice that contains some fraud.

In [ ]:
RUN_PREPROCESSING_COMPARISON = False   # set to True to rerun (5-15 minutes)


def prepare(option):
    train, test = X_train_normal.copy(), X_test.copy()
    if option == "a":
        scaler = StandardScaler().fit(train)
        return scaler.transform(train), scaler.transform(test)
    if option == "b":
        scaler = RobustScaler().fit(train[["Amount"]])
        train["Amount"] = scaler.transform(train[["Amount"]]).ravel()
        test["Amount"] = scaler.transform(test[["Amount"]]).ravel()
        return train.values, test.values
    if option == "c":
        train["Amount"], test["Amount"] = np.log1p(train["Amount"]), np.log1p(test["Amount"])
        scaler = StandardScaler().fit(train)
        return scaler.transform(train), scaler.transform(test)


def train_and_score(option, seed, epochs=20):
    torch.manual_seed(seed)
    np.random.seed(seed)
    train, test = prepare(option)
    train = torch.tensor(train, dtype=torch.float32)
    test = torch.tensor(test, dtype=torch.float32)
    model = nn.Sequential(
        nn.Linear(29, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(),
        nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 29),
    )
    optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    loader = torch.utils.data.DataLoader(train, batch_size=256, shuffle=True)
    for _ in range(epochs):
        for batch in loader:
            optimiser.zero_grad()
            loss_fn(model(batch), batch).backward()
            optimiser.step()
    with torch.no_grad():
        errors = ((model(test) - test) ** 2).mean(dim=1).numpy()
    return average_precision_score(y_test, errors)


if RUN_PREPROCESSING_COMPARISON:
    results = []
    for option in ["a", "b", "c"]:
        for seed in [42, 1, 7]:
            pr_auc = train_and_score(option, seed)
            print(f"option {option}, seed {seed}: PR-AUC = {pr_auc:.3f}")
            results.append({"option": option, "seed": seed, "pr_auc": pr_auc})
    display(pd.DataFrame(results).pivot(index="option", columns="seed", values="pr_auc").round(3))
else:
    print("Skipped (see the results table above). Set RUN_PREPROCESSING_COMPARISON = True to rerun.")

## 4. A validation set: split first, then fit anything

During training we want a **practice exam**: rows the network never trains on,
to check whether it's learning real patterns or just memorising. The test set
is the final exam and stays untouched until the end.

So 10% of the normal training rows become a **validation set**. The order
matters: **split first, then fit the scaler** on the training part only. If
the scaler saw the validation rows, information would leak into the practice
exam.

The validation set contains **no fraud**, so it can't measure fraud
detection or choose a threshold. What it *can* show is the reconstruction error
on unseen normal rows. If training loss keeps falling while validation loss
rises, the network is **overfitting**.

In [ ]:
X_tr, X_val = train_test_split(X_train_normal, test_size=0.1, random_state=RANDOM_STATE)

scaler = RobustScaler().fit(X_tr[["Amount"]])   # median and IQR from the training part only

X_tr_scaled, X_val_scaled, X_test_scaled = X_tr.copy(), X_val.copy(), X_test.copy()
X_tr_scaled["Amount"] = scaler.transform(X_tr[["Amount"]]).ravel()
X_val_scaled["Amount"] = scaler.transform(X_val[["Amount"]]).ravel()
X_test_scaled["Amount"] = scaler.transform(X_test[["Amount"]]).ravel()

print("train:", X_tr_scaled.shape, " val:", X_val_scaled.shape, " test:", X_test_scaled.shape)
print("median:", scaler.center_, " IQR:", scaler.scale_)

## 5. Tensors and batches

PyTorch works with **tensors**, which are tables of numbers much like NumPy
arrays, stored as 32-bit floats (the standard for neural networks). A
**DataLoader** serves the training rows in shuffled **batches** of 256: the
network looks at 256 rows, adjusts itself a little, then moves on.

203,941 rows ÷ 256 = 796.6, so each **epoch** (one full pass over the data) is
796 full batches plus one final batch of 165 rows: **797 steps**. Twenty
epochs means 15,940 small adjustments.

In [ ]:
X_tr_t = torch.tensor(X_tr_scaled.values, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled.values, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled.values, dtype=torch.float32)

print(X_tr_t.shape, X_tr_t.dtype)
print("batches per epoch:", int(np.ceil(len(X_tr_t) / 256)))

## 6. The network

```
29  →  16  →  8  →  16  →  29
  └ encoder ┘  ↑   └ decoder ┘
           bottleneck
```

- **`nn.Linear(a, b)`** produces `b` outputs, each a weighted mix of the `a`
  inputs plus a bias. The weights and biases are the **parameters**, the knobs
  training adjusts: `a × b + b` per layer, **1,253** in total. That's tiny, far
  too few to memorise 203,941 rows, which is exactly the point.
- **`nn.ReLU()`** turns negatives into 0. Without it, stacked Linear layers
  collapse into one Linear layer and can only learn straight-line relationships.
- **No ReLU on the last layer.** The rebuilt values must be able to go negative
  (V1 goes down to −56), and a final ReLU would clamp them at 0.

In [ ]:
def build_autoencoder():
    return nn.Sequential(
        nn.Linear(29, 16), nn.ReLU(),   # encoder
        nn.Linear(16, 8), nn.ReLU(),    # bottleneck
        nn.Linear(8, 16), nn.ReLU(),    # decoder
        nn.Linear(16, 29),              # rebuilt row (no activation)
    )


print(build_autoencoder())
print("parameters:", sum(p.numel() for p in build_autoencoder().parameters()))

## 7. Training

- **Loss: MSE.** The average of (original − rebuilt)². This is the
  reconstruction error. Note that the batch is both the input *and* the
  target: `loss_fn(model(batch), batch)`. That's what makes it an autoencoder.
- **Optimiser: Adam** with learning rate 0.001, the standard starting point.
- Every step runs the same four lines: clear old gradients → rebuild and measure
  the error → work out which way each knob should turn (`backward`) → turn
  them a little (`step`).

In [ ]:
torch.manual_seed(RANDOM_STATE)
model = build_autoencoder()
loss_fn = nn.MSELoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader = torch.utils.data.DataLoader(X_tr_t, batch_size=256, shuffle=True)

history = []
for epoch in range(20):
    total = 0.0
    for batch in train_loader:
        optimiser.zero_grad()
        loss = loss_fn(model(batch), batch)
        loss.backward()
        optimiser.step()
        total += loss.item() * len(batch)
    train_loss = total / len(X_tr_t)

    with torch.no_grad():
        val_loss = loss_fn(model(X_val_t), X_val_t).item()

    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})
    print(f"epoch {epoch + 1:2d}: train loss = {train_loss:.4f}   val loss = {val_loss:.4f}")

In [ ]:
hist = pd.DataFrame(history)
plt.figure(figsize=(8, 4))
plt.plot(hist["epoch"], hist["train_loss"], marker="o", label="training loss")
plt.plot(hist["epoch"], hist["val_loss"], marker="o", label="validation loss")
plt.xlabel("epoch")
plt.ylabel("MSE (reconstruction error)")
plt.title("Autoencoder training: loss per epoch")
plt.xticks(range(1, 21))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Reading the curves:** both losses drop steeply for about 5 epochs, then
**plateau**. That means learning has mostly finished, so more epochs would add
little.

This is **not overfitting**. Overfitting would show the lines *splitting
apart*: training loss still falling while validation loss turns upward, like a
student who memorises past-paper answers and then does worse on a new mock
exam. Here validation loss keeps falling (or stays flat) right to the end.

Validation loss sits slightly *below* training loss because the training loss
is averaged over the whole epoch, including early batches when the network
was still worse. Validation is measured once, at the end of the epoch.

## 8. Score the test set

Each test transaction's **reconstruction error is its risk score**. The network
has never seen a fraud, so frauds should be rebuilt badly. We compare medians,
not means, because a few extreme legitimate rows would drag a mean around.

In [ ]:
with torch.no_grad():
    errors = ((model(X_test_t) - X_test_t) ** 2).mean(dim=1).numpy()

print("normal median error:", round(float(np.median(errors[y_test == 0])), 3))
print("fraud  median error:", round(float(np.median(errors[y_test == 1])), 3))

pr_auc = average_precision_score(y_test, errors)
print(f"PR-AUC: {pr_auc:.4f}  ({pr_auc / y_test.mean():.0f}x the no-skill baseline of {y_test.mean():.4f})")

The typical fraud is rebuilt about **30 times worse** than the typical normal
transaction. That's evidence the mechanism works: the error separates fraud
from normal without the model ever being told what fraud is.

PR-AUC is a summary across *every* possible threshold. To know what a fraud
team would actually experience, we have to draw a line.

## 9. Threshold and confusion matrix

The threshold is chosen **exactly as for the Isolation Forest**: the one that
maximises F1 on the test set. That makes the absolute numbers slightly
optimistic (the line was drawn while looking at the exam), but both models get
the same advantage, so the comparison is fair.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, errors)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best = int(f1[:-1].argmax())
threshold = thresholds[best]

y_pred = (errors >= threshold).astype(int)   # at or above the line = flagged
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"threshold: {threshold:.4f}")
print(f"precision: {precision[best]:.3f}   recall: {recall[best]:.3f}   F1: {f1[best]:.3f}")
print(f"TP={tp}  FN={fn}  FP={fp}  TN={tn}")

ConfusionMatrixDisplay(cm, display_labels=["Normal", "Fraud"]).plot(cmap="Blues", colorbar=False, values_format=",d")
plt.title("Autoencoder: confusion matrix (test set)")
plt.show()

## 10. Side by side with the Isolation Forest

The Isolation Forest is retrained here with exactly the settings from
`02_isolation_forest.ipynb`, so both models can be compared fraud by fraud.

In [ ]:
iso = IsolationForest(n_estimators=200, max_samples=256, contamination="auto",
                      random_state=RANDOM_STATE, n_jobs=-1).fit(X_train_normal)   # raw data: scaling does not affect Isolation Forest
if_scores = -iso.score_samples(X_test)

p_if, r_if, t_if = precision_recall_curve(y_test, if_scores)
f1_if = 2 * p_if * r_if / (p_if + r_if + 1e-12)
best_if = int(f1_if[:-1].argmax())
if_pred = (if_scores >= t_if[best_if]).astype(int)
tn_i, fp_i, fn_i, tp_i = confusion_matrix(y_test, if_pred).ravel()

def summary(pr, p, r, f, tp_, fn_, fp_):
    return [round(float(v), 3) for v in (pr, p, r, f)] + [int(tp_), int(fn_), int(fp_)]

pd.DataFrame({
    "Isolation Forest": summary(average_precision_score(y_test, if_scores), p_if[best_if], r_if[best_if], f1_if[best_if], tp_i, fn_i, fp_i),
    "Autoencoder": summary(pr_auc, precision[best], recall[best], f1[best], tp, fn, fp),
}, index=["PR-AUC", "precision", "recall", "F1", "frauds caught (TP)", "frauds missed (FN)", "false alarms (FP)"], dtype=object)

In [ ]:
plt.figure(figsize=(6.5, 5))
plt.plot(r_if, p_if, label=f"Isolation Forest (PR-AUC {average_precision_score(y_test, if_scores):.3f})")
plt.plot(recall, precision, label=f"Autoencoder (PR-AUC {pr_auc:.3f})")
plt.axhline(y_test.mean(), linestyle=":", color="grey", label="no-skill baseline")
plt.xlabel("Recall (share of fraud caught)")
plt.ylabel("Precision (share of flags that were real fraud)")
plt.title("Precision-Recall curves, same test set")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The autoencoder catches **about twice as many frauds** while raising **about
half as many false alarms**. Precision rises from roughly 1 flag in 5 being
real fraud to roughly 1 in 2.

Usually there's a trade-off: catch more frauds and you raise more false
alarms. Improving *both at once* means the autoencoder simply **ranks**
transactions better, with real frauds higher up its list. That's what the
higher PR-AUC says.

## 11. Do the two models catch *different* frauds?

If they did, combining them might help. So let's compare fraud by fraud.

In [ ]:
fraud = (y_test == 1).values
ae_caught = (y_pred == 1) & fraud
if_caught = (if_pred == 1) & fraud

print("IF caught:", if_caught.sum(), "  AE caught:", ae_caught.sum())
print("caught by both:   ", (ae_caught & if_caught).sum())
print("only autoencoder: ", (ae_caught & ~if_caught).sum())
print("only IF:          ", (if_caught & ~ae_caught).sum())
print("caught by neither:", (fraud & ~ae_caught & ~if_caught).sum())

The autoencoder catches almost every fraud the Isolation Forest catches, plus
many more. Combining them would add only a couple of frauds, so the
autoencoder is an **upgrade, not a partner**.

The most telling number is **caught by neither**: over 40 of the 95 frauds
look *normal* to both models. No method that only learns "what normal looks
like" can see them. Catching those needs a model that learns from labelled
fraud examples, a **supervised** model (a stretch goal for this project).

## 12. Limitations, stated plainly

- **The threshold was chosen on the test set**, as for the Isolation Forest.
  The numbers are slightly optimistic; a production system would set the
  threshold on a separate labelled slice.
- **The preprocessing choice was also compared on the test set** (section 3).
- **Results vary with the random seed** (PR-AUC about 0.38–0.44 across seeds in the
  comparison), so the exact figures shouldn't be over-read.
- **Unusual ≠ fraud.** Extreme legitimate transactions get high errors and
  become false alarms, and frauds that look normal are missed.

## 13. Save the model

The API will need **three things that belong together**: the network's
parameters, the exact scaler (a new transaction's Amount must be scaled with
the same median and IQR), and the threshold. The threshold belongs to *this*
trained model: another training run produces slightly different errors and a
different best line. So all three are saved as one package. Typing a number in
by hand later would risk using one model's line on another model's scores.

(`models/` is gitignored: artifacts are regenerated by running
`python -m src.models.train_autoencoder`.)

In [ ]:
os.makedirs("../models", exist_ok=True)
torch.save(model.state_dict(), "../models/autoencoder.pt")
joblib.dump(scaler, "../models/autoencoder_scaler.joblib")
with open("../models/autoencoder_config.json", "w") as f:
    json.dump({"layers": [29, 16, 8, 16, 29], "features": feature_cols,
               "scaled_features": ["Amount"], "threshold": float(threshold)}, f, indent=2)

# Check: a freshly built network loaded from disk gives identical scores
loaded = build_autoencoder()
loaded.load_state_dict(torch.load("../models/autoencoder.pt"))
with torch.no_grad():
    errors_reloaded = ((loaded(X_test_t) - X_test_t) ** 2).mean(dim=1).numpy()
print("identical scores after reload:", np.allclose(errors_reloaded, errors))